In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.preprocessing import StandardScaler

# Display all columns
pd.set_option('display.max_columns', None)

In [4]:
# Load the dataset

df = pd.read_csv("../youtube_data.csv")


In [5]:
print("First 5 Rows")
display(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nColumn Names")
print(df.columns.tolist())



First 5 Rows


,video_id,title,description,published_date,channel_id,channel_title,tags,category_id,view_count,like_count,comment_count,duration,thumbnail
0,gsJAlLOFBv0,TINY Tech That Actually Works,No description available,2025-05-02T17:37:10Z,UCMiJRAwDNSNzuYeN2uWa0pA,Mrwhosetheboss,"['tiny', 'tech', 'gadgets', 'small', 'miniature']",28,8962092.0,243350.0,515.0,PT57S,https://i.ytimg.com/vi/gsJAlLOFBv0/default.jpg
1,ypicIkaiViM,AI & future of workforce: Andrew Yang on how t...,"Andrew Yang, Forward Party co-chair and former...",2025-06-18T12:39:53Z,UCrp_UI8XtuYfpiqluWLD7Lw,CNBC Television,"['Squawk Box U.S.', 'CNBC', 'business news', '...",25,289626.0,3393.0,1240.0,PT7M50S,https://i.ytimg.com/vi/ypicIkaiViM/default.jpg
2,1Nef8LPO-jo,5 ILLEGAL gadgets that will get you ARRESTED,#shorts #technology \n\nI spend a LOT of time ...,2022-11-01T11:00:06Z,UCMiJRAwDNSNzuYeN2uWa0pA,Mrwhosetheboss,"['shorts', 'tech']",28,81372201.0,4178447.0,6378.0,PT47S,https://i.ytimg.com/vi/1Nef8LPO-jo/default.jpg
3,lCHqmzynO-s,Overrated vs. Underrated Tech,💬 Join my Discord server: https://discord.gg/g...,2024-07-08T18:04:31Z,UCPk2s5c4R_d-EUUNvFFODoA,Gohar Khan,"['thailand', 'surin', 'style', 'travel', 'day'...",27,21255964.0,909386.0,2681.0,PT31S,https://i.ytimg.com/vi/lCHqmzynO-s/default.jpg
4,7uFrtqSwYzM,APPLE Glass Revolutionizes AR Experience Forever!,Discover the revolutionary world of augmented ...,2024-12-22T16:49:00Z,UCxqG_E-68WAE0TWYfIopv6Q,Digifix,"['apple glasses price', 'apple glasses design'...",28,2790436.0,44278.0,1359.0,PT16S,https://i.ytimg.com/vi/7uFrtqSwYzM/default.jpg



Dataset Shape
(600, 13)

Column Names
['video_id', 'title', 'description', 'published_date', 'channel_id', 'channel_title', 'tags', 'category_id', 'view_count', 'like_count', 'comment_count', 'duration', 'thumbnail']


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        600 non-null    str    
 1   title           600 non-null    str    
 2   description     600 non-null    str    
 3   published_date  600 non-null    str    
 4   channel_id      600 non-null    str    
 5   channel_title   600 non-null    str    
 6   tags            600 non-null    str    
 7   category_id     600 non-null    int64  
 8   view_count      600 non-null    float64
 9   like_count      600 non-null    float64
 10  comment_count   600 non-null    float64
 11  duration        600 non-null    str    
 12  thumbnail       600 non-null    str    
dtypes: float64(3), int64(1), str(9)
memory usage: 61.1 KB


In [7]:
# Display summary statistics of numerical columns

df.describe()

,category_id,view_count,like_count,comment_count
count,600.000000,6.000000e+02,6.000000e+02,600.000000
mean,24.903333,8.080299e+06,2.174646e+05,1970.144781
std,4.863477,2.455377e+07,5.074207e+05,4421.455908
min,1.000000,3.120000e+02,0.000000e+00,0.000000
25%,22.000000,8.150100e+04,1.646000e+03,13.000000
50%,27.000000,9.587180e+05,2.214900e+04,252.500000
75%,28.000000,5.991357e+06,2.174646e+05,1802.500000
max,30.000000,3.437590e+08,4.421091e+06,40241.000000


In [8]:
# Check for missing values in each column

df.isnull().sum()


video_id          0
title             0
description       0
published_date    0
channel_id        0
channel_title     0
tags              0
category_id       0
view_count        0
like_count        0
comment_count     0
duration          0
thumbnail         0
dtype: int64

In [9]:
# Count total missing values

total_missing = df.isnull().sum().sum()

print("Total Missing Values:", total_missing)

Total Missing Values: 0


If Missing values more than 10-20.

df.fillna(df.median(numeric_only=True), inplace=True)

In [10]:
df = df.dropna()

In [11]:
# Check duplicate rows

duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 16


In [12]:
# Remove duplicate rows

df = df.drop_duplicates()

In [13]:
print("Duplicate Rows After Cleaning:", df.duplicated().sum())

Duplicate Rows After Cleaning: 0


In [14]:

pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'

invalid = df[~df["duration"].astype(str).str.match(pattern, na=False)]

print(invalid[["duration"]])
print("Number of invalid rows:", len(invalid))

    duration
502      P0D
Number of invalid rows: 1


In [15]:
def convert_duration(duration):
    if pd.isna(duration):
        return 0

    duration = str(duration).strip()

    pattern = r'^PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?$'
    match = re.match(pattern, duration)

    if match is None:
        return 0

    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)

    return hours * 3600 + minutes * 60 + seconds

df["duration_seconds"] = df["duration"].astype(str).apply(convert_duration)

In [16]:
df[["duration","duration_seconds"]].head(20)

,duration,duration_seconds
0,PT57S,57
1,PT7M50S,470
2,PT47S,47
3,PT31S,31
4,PT16S,16
5,PT56S,56
6,PT1M,60
7,PT30S,30
8,PT7M17S,437
9,PT1M,60


In [17]:

# Remove unnecessary columns

columns_to_drop = [
    "video_id",
    "title",
    "description",
    "published_date",
    "channel_id",
    "channel_title",
    "tags",
    "thumbnail"
]

df.drop(columns=columns_to_drop, inplace=True)

In [18]:

print(df.columns.tolist())

['category_id', 'view_count', 'like_count', 'comment_count', 'duration', 'duration_seconds']


In [19]:
scaler = StandardScaler()

numerical_columns = [
    "view_count",
    "comment_count",
    "duration_seconds",
    "category_id"
]

df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

In [20]:
display(df.head())

print(df.describe())

,category_id,view_count,like_count,comment_count,duration,duration_seconds
0,0.634019,0.029288,243350.0,-0.330080,PT57S,-0.329112
1,0.017250,-0.319920,3393.0,-0.165235,PT7M50S,0.044432
2,0.634019,2.944978,4178447.0,1.003008,PT47S,-0.338157
3,0.428429,0.524317,909386.0,0.162410,PT31S,-0.352628
4,0.634019,-0.219222,44278.0,-0.138177,PT16S,-0.366195


        category_id    view_count    like_count  comment_count  \
count  5.840000e+02  5.840000e+02  5.840000e+02   5.840000e+02   
mean   2.676702e-16  4.866731e-17  2.204846e+05  -3.650048e-17   
std    1.000857e+00  1.000857e+00  5.135630e+05   1.000857e+00   
min   -4.916902e+00 -3.315698e-01  0.000000e+00  -4.471773e-01   
25%   -5.995192e-01 -3.278677e-01  1.646000e+03  -4.437668e-01   
50%    4.284292e-01 -2.916785e-01  2.214900e+04  -3.871509e-01   
75%    6.340188e-01 -8.035010e-02  2.174646e+05  -3.358575e-02   
max    1.045198e+00  1.351033e+01  4.421091e+06   8.702542e+00   

       duration_seconds  
count      5.840000e+02  
mean      -1.216683e-17  
std        1.000857e+00  
min       -3.806666e-01  
25%       -3.643862e-01  
50%       -3.395134e-01  
75%        1.842847e-02  
max        1.513090e+01  


In [21]:
# Save the cleaned dataset

df.to_csv("../processed/cleaned_youtube_data.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
